# Lecture 3 — Class Exercise
#Line Charts & Slopegraphs: CO2 Emissions

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

df = pd.read_csv('co2_emissions.csv')
print(f"Loaded: {len(df)} rows | Countries: {df['Country'].nunique()} | Years: {df['Year'].min()}-{df['Year'].max()}")
print(df.head())

Loaded: 345 rows | Countries: 15 | Years: 2000-2022
         Country         Region  Year  CO2_Mt  CO2_per_capita
0  United States  North America  2000  5857.6            1.32
1  United States  North America  2001  5724.0            1.26
2  United States  North America  2002  5652.8            1.11
3  United States  North America  2003  5592.8            1.29
4  United States  North America  2004  5743.2            1.12


In [ ]:
print("Countries:", df['Country'].unique())
print("\nCO2 range:", df['CO2_Mt'].min(), "to", df['CO2_Mt'].max(), "Mt")
print("\nRegional averages (2022):")
print(df[df['Year']==2022].groupby('Region')['CO2_Mt'].mean().sort_values(ascending=False).round(1))

Countries: ['United States' 'China' 'India' 'Germany' 'United Kingdom' 'France'
 'Brazil' 'Japan' 'Canada' 'Australia' 'South Korea' 'Russia'
 'South Africa' 'Mexico' 'Indonesia']

CO2 range: 125.3 to 12409.5 Mt

Regional averages (2022):
Region
Asia             3531.1
North America    2393.8
Latin America     629.2
Africa            534.4
Europe            496.5
Oceania           493.7
Name: CO2_Mt, dtype: float64


# Task 1 — Multi-series Line Chart

In [ ]:
asia = df[df['Region'] == 'Asia'].copy()
highlight = 'China'

In [ ]:
fig = go.Figure()
for country in asia['Country'].unique():
    d = asia[asia['Country'] == country]

    if country == highlight:
        fig.add_trace(go.Scatter(
            x=d['Year'],
            y=d['CO2_Mt'],
            mode='lines',
            name=country,
            line=dict(color='#2E75B6', width=3)
        ))
    else:
        fig.add_trace(go.Scatter(
            x=d['Year'],
            y=d['CO2_Mt'],
            mode='lines',
            name=country,
            line=dict(color='#DDDDDD', width=1),
            showlegend=False
        ))

In [ ]:
last = asia[(asia['Country'] == highlight) & (asia['Year'] == asia['Year'].max())]

fig.add_annotation(
    x=last['Year'].values[0],
    y=last['CO2_Mt'].values[0],
    text=f"<b>{highlight}</b>",
    showarrow=False,
    xanchor='left',
    xshift=6,
    font=dict(color='#2E75B6', size=12, family='Arial')
)

In [ ]:
#declutter
fig.update_layout(
    title="China’s CO2 emissions surged since 2000 — no other Asian country is close",
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family='Arial', size=13),

    yaxis=dict(
        title='CO2 Emissions (Mt)',
        gridcolor='#EEEEEE'
    ),

    xaxis=dict(
        showgrid=False,
        title=''
    ),

    margin=dict(l=60, r=80, t=60, b=40)
)

In [ ]:
fig.show()

# Task 2 — Slopegraph: Regional Change 2000 vs 2022

In [ ]:
regional = df.groupby(['Region', 'Year'])['CO2_Mt'].mean().reset_index()
sg = regional[regional['Year'].isin([2000, 2022])].copy()

In [ ]:
changes = sg.groupby('Region')['CO2_Mt'].agg(['first', 'last'])

color_map = {
    region: '#2E75B6' if row['last'] > row['first'] else '#F28E2B'
    for region, row in changes.iterrows()
}

In [ ]:
fig = px.line(
    sg.sort_values('Year'),
    x='Year',
    y='CO2_Mt',
    color='Region',
    color_discrete_map=color_map,
    markers=True
)

In [ ]:
for region in sg['Region'].unique():
    d = sg[sg['Region'] == region].sort_values('Year')

    fig.update_traces(
        selector=dict(name=region),
        mode='lines+markers+text',
        text=[
            f"{d['CO2_Mt'].iloc[0]:.0f}",
            f"{region}<br>{d['CO2_Mt'].iloc[1]:.0f}"
        ],
        textposition=['middle left', 'middle right'],
        textfont=dict(size=10, family='Arial'),
        showlegend=False
    )

In [ ]:
fig.update_layout(
    title="Asia shows the strongest growth in emissions since 2000; Europe declines",
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family='Arial', size=12),

    xaxis=dict(
        tickvals=[2000, 2022],
        ticktext=['2000', '2022'],
        showgrid=False,
        range=[1995, 2028]
    ),

    yaxis=dict(
        showgrid=False,
        showticklabels=False,
        title=''
    ),

    margin=dict(l=80, r=120, t=60, b=40)
)

In [ ]:
fig.show()